# 14 — Préparation du corpus Facebook candidat

Ce notebook transforme les publications collectées en documents RAG propres, mais uniquement dans `data/experiments/`. Il normalise les caractères décoratifs Unicode, retire les textes trop courts et conserve seulement la republication la plus récente lorsqu’un même texte apparaît plusieurs fois.

Il produit ensuite des chunks candidats avec le chunker validé. **Aucun index LanceDB ou BM25 de production n’est modifié.**

In [1]:
from pathlib import Path
import hashlib
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

def fingerprint_path(path):
    path = Path(path)
    digest = hashlib.sha256()
    if path.is_file():
        digest.update(path.read_bytes())
    elif path.is_dir():
        for child in sorted(p for p in path.rglob('*') if p.is_file()):
            digest.update(str(child.relative_to(path)).encode('utf-8'))
            digest.update(child.read_bytes())
    return digest.hexdigest()

print('Projet :', PROJECT_ROOT)

Projet : c:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs\RAG ENSEA


In [2]:
import importlib
import facebook_processor as facebook_processor_module
from config import load_config

facebook_processor_module = importlib.reload(facebook_processor_module)
FacebookPostProcessor = facebook_processor_module.FacebookPostProcessor
config = load_config(PROJECT_ROOT / 'config.yaml')
settings = config.facebook_scraping
processor = FacebookPostProcessor(config)

production_cleaned = config.project.data_path / 'cleaned'
production_chunks = config.project.data_path / 'chunks' / 'chunks.jsonl'
before_cleaned = fingerprint_path(production_cleaned)
before_chunks = fingerprint_path(production_chunks)

print('Entrée :', settings.raw_full_path)
print('Sortie candidate :', settings.cleaned_candidate_full_path)
print('Minimum RAG :', settings.min_rag_characters, 'caractères')

Entrée : <PROJECT_ROOT>\data\facebook\raw
Sortie candidate : <PROJECT_ROOT>\data\experiments\facebook_cleaned_candidate
Minimum RAG : 80 caractères


In [3]:
processing_report = processor.process_all()
processing_report

{'raw_posts': 232,
 'removed_too_short': 2,
 'removed_duplicate_content': 3,
 'selected_posts': 227,
 'per_page': {'AES_page': 28,
  'CDR_page': 30,
  'DAS_page': 26,
  'DISE_page': 30,
  'ECOSTAT_page': 29,
  'EDSC_page': 29,
  'EJS_page': 28,
  'ENSEA_page': 27},
 'cleaned_candidate_dir': 'C:\\Users\\Ariel\\OneDrive - ENSEA\\Documents\\LLMs\\RAG ENSEA\\data\\experiments\\facebook_cleaned_candidate',
 'written_documents': 227,
 'production_cleaned_dir': 'C:\\Users\\Ariel\\OneDrive - ENSEA\\Documents\\LLMs\\RAG ENSEA\\data\\cleaned',
 'report_path': 'C:\\Users\\Ariel\\OneDrive - ENSEA\\Documents\\LLMs\\RAG ENSEA\\data\\experiments\\facebook_processing_report.json'}

In [4]:
import frontmatter
import pandas as pd

candidate_files = sorted(settings.cleaned_candidate_full_path.glob('facebook_*.md'))
documents = []
for path in candidate_files:
    post = frontmatter.load(path)
    documents.append({**dict(post.metadata), 'text': post.content, 'file': path.name})
docs_df = pd.DataFrame(documents)

summary = pd.DataFrame({
    'documents': docs_df.groupby('page_key').size(),
    'median_chars': docs_df.groupby('page_key')['text'].apply(lambda s: int(s.str.len().median())),
    'min_chars': docs_df.groupby('page_key')['text'].apply(lambda s: int(s.str.len().min())),
    'max_chars': docs_df.groupby('page_key')['text'].apply(lambda s: int(s.str.len().max())),
})
display(summary)
display(docs_df[['page_key', 'date', 'title', 'url', 'text']].head(8))

,documents,median_chars,min_chars,max_chars
page_key,,,,
AES_page,28,646,234,1338
CDR_page,30,738,375,1124
DAS_page,26,848,181,2088
DISE_page,30,1361,392,2478
ECOSTAT_page,29,700,410,2016
EDSC_page,29,437,236,1365
EJS_page,28,815,152,1144
ENSEA_page,27,1227,349,2747


,page_key,date,title,url,text
0,DAS_page,2026-06-27T17:59:23+00:00,DivisionAS — La Division des Analystes Statist...,https://www.facebook.com/DivisionAS/posts/pfbi...,# DivisionAS — La Division des Analystes Stati...
1,DAS_page,2026-06-28T12:38:02+00:00,DivisionAS — La Division des Analystes Statist...,https://www.facebook.com/DivisionAS/posts/pfbi...,# DivisionAS — La Division des Analystes Stati...
2,DAS_page,2026-07-10T13:21:38+00:00,DivisionAS — La Division des Analystes Statist...,https://www.facebook.com/DivisionAS/posts/pfbi...,# DivisionAS — La Division des Analystes Stati...
3,DAS_page,2026-07-24T19:30:25+00:00,DivisionAS — ✨ Le rideau se lève... ✨,https://www.facebook.com/DivisionAS/posts/pfbi...,# DivisionAS — ✨ Le rideau se lève... ✨\n\n✨ L...
4,DAS_page,2026-07-28T12:24:20+00:00,DivisionAS — 🚀 STAT-HACK’26 : l’excellence de ...,https://www.facebook.com/DivisionAS/posts/pfbi...,# DivisionAS — 🚀 STAT-HACK’26 : l’excellence d...
5,DAS_page,2026-07-29T13:18:42+00:00,DivisionAS — 🎉 Félicitations à DEMGNE TCHUDJO ...,https://www.facebook.com/DivisionAS/posts/pfbi...,# DivisionAS — 🎉 Félicitations à DEMGNE TCHUDJ...
6,ECOSTAT_page,2025-07-15T12:32:49+00:00,DepecheEcostat — Dépêche N°17,https://www.facebook.com/DepecheEcostat/posts/...,# DepecheEcostat — Dépêche N°17\n\nDépêche N°1...
7,ECOSTAT_page,2025-08-12T21:10:36+00:00,DepecheEcostat — Dépêche n°18,https://www.facebook.com/DepecheEcostat/posts/...,# DepecheEcostat — Dépêche n°18\n\nDépêche n°1...


In [5]:
expected_selected = (
    processing_report['raw_posts']
    - processing_report.get('removed_too_short', 0)
    - processing_report.get('removed_missing_provenance', 0)
    - processing_report.get('removed_duplicate_content', 0)
)
assert len(docs_df) == processing_report['selected_posts'] == expected_selected
assert docs_df['content_hash'].is_unique, 'Des textes identiques restent présents.'
assert docs_df['url'].notna().all() and docs_df['date'].notna().all()
assert docs_df['source'].eq('facebook').all()
assert docs_df['doc_type'].eq('facebook_post').all()
assert docs_df['text'].str.len().ge(settings.min_rag_characters).all()
assert not docs_df['text'].str.contains('\u20e3', regex=False).any(), 'Des chiffres-émojis subsistent.'
print('Documents candidats validés :', len(docs_df))

Documents candidats validés : 227


## Chunking candidat

On applique exactement le chunker hiérarchique validé, mais avec des chemins de sortie expérimentaux.

In [6]:
import chunker as chunker_module

chunker_module = importlib.reload(chunker_module)
HierarchicalChunker = chunker_module.HierarchicalChunker
candidate_chunker = HierarchicalChunker(config)
facebook_chunks = candidate_chunker.chunk_all(
    cleaned_dir=settings.cleaned_candidate_full_path,
    output_path=settings.chunks_candidate_full_path,
)
print('Documents :', len(docs_df))
print('Chunks candidats :', len(facebook_chunks))
print('Sortie :', settings.chunks_candidate_full_path)

Documents : 227
Chunks candidats : 250
Sortie : <PROJECT_ROOT>\data\experiments\facebook_chunks_candidate.jsonl


In [7]:
chunk_df = pd.DataFrame([chunk.__dict__ for chunk in facebook_chunks])
chunk_quality = pd.Series({
    'chunks': len(chunk_df),
    'documents_represented': chunk_df['url'].nunique(),
    'median_child_chars': int(chunk_df['text'].str.len().median()),
    'max_child_chars': int(chunk_df['text'].str.len().max()),
    'median_parent_chars': int(chunk_df['parent_text'].str.len().median()),
    'max_parent_chars': int(chunk_df['parent_text'].str.len().max()),
    'duplicate_chunk_ids': int(chunk_df['chunk_id'].duplicated().sum()),
})
display(chunk_quality.to_frame('valeur'))
display(chunk_df[['title', 'date', 'category', 'url', 'text']].head(8))

,valeur
chunks,250
documents_represented,227
median_child_chars,760
max_child_chars,1593
median_parent_chars,830
max_parent_chars,2753
duplicate_chunk_ids,0


,title,date,category,url,text
0,DivisionAS — La Division des Analystes Statist...,2026-06-27T17:59:23+00:00,facebook/DAS_page,https://www.facebook.com/DivisionAS/posts/pfbi...,# DivisionAS — La Division des Analystes Stati...
1,DivisionAS — La Division des Analystes Statist...,2026-06-28T12:38:02+00:00,facebook/DAS_page,https://www.facebook.com/DivisionAS/posts/pfbi...,# DivisionAS — La Division des Analystes Stati...
2,DivisionAS — La Division des Analystes Statist...,2026-07-10T13:21:38+00:00,facebook/DAS_page,https://www.facebook.com/DivisionAS/posts/pfbi...,# DivisionAS — La Division des Analystes Stati...
3,DivisionAS — ✨ Le rideau se lève... ✨,2026-07-24T19:30:25+00:00,facebook/DAS_page,https://www.facebook.com/DivisionAS/posts/pfbi...,# DivisionAS — ✨ Le rideau se lève... ✨ \n✨ L...
4,DivisionAS — 🚀 STAT-HACK’26 : l’excellence de ...,2026-07-28T12:24:20+00:00,facebook/DAS_page,https://www.facebook.com/DivisionAS/posts/pfbi...,# DivisionAS — 🚀 STAT-HACK’26 : l’excellence d...
5,DivisionAS — 🚀 STAT-HACK’26 : l’excellence de ...,2026-07-28T12:24:20+00:00,facebook/DAS_page,https://www.facebook.com/DivisionAS/posts/pfbi...,#STATHACK26 #DAS #AnalysteStatisticien #DataSc...
6,DivisionAS — 🎉 Félicitations à DEMGNE TCHUDJO ...,2026-07-29T13:18:42+00:00,facebook/DAS_page,https://www.facebook.com/DivisionAS/posts/pfbi...,# DivisionAS — 🎉 Félicitations à DEMGNE TCHUDJ...
7,DepecheEcostat — Dépêche N°17,2025-07-15T12:32:49+00:00,facebook/ECOSTAT_page,https://www.facebook.com/DepecheEcostat/posts/...,# DepecheEcostat — Dépêche N°17 \nDépêche N°1...


In [8]:
assert not chunk_df.empty
assert chunk_df['chunk_id'].is_unique
assert chunk_df['doc_type'].eq('facebook_post').all()
assert chunk_df['url'].nunique() == len(docs_df)
assert fingerprint_path(production_cleaned) == before_cleaned, 'Le corpus nettoyé principal a changé.'
assert fingerprint_path(production_chunks) == before_chunks, 'Les chunks principaux ont changé.'
print('Corpus Facebook candidat validé ; corpus institutionnel intact.')

Corpus Facebook candidat validé ; corpus institutionnel intact.


## Arrêt de validation

Sauvegardez le notebook et transmettez les deux tableaux de synthèse. Aucun embedding ni index n’est construit à cette étape.